In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
'''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
'''
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

"\nfor dirname, _, filenames in os.walk('/kaggle/input'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n"

In [3]:
df= pd.read_csv("2023_tesla_stocks_news_mixed.csv")

In [6]:
df = df.drop(columns=["Unnamed: 0"])
df

,timestamp,symbol,open,high,low,close,volume,sma_10,sma_50,ema_20,...,scores_13,scores_14,scores_15,scores_16,scores_17,scores_18,scores_19,scores_20,scores_21,scores_22
0,2023-01-03 09:00:00,TSLA,120.02,121.23,115.94,118.52,1306706,118.5200,118.5200,118.520000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-03 10:00:00,TSLA,118.53,120.50,118.20,120.38,568635,119.4500,119.4500,118.697143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-03 11:00:00,TSLA,120.28,120.50,117.95,118.61,547403,119.1700,119.1700,118.688844,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-03 12:00:00,TSLA,118.63,118.99,116.61,118.28,1529445,118.9475,118.9475,118.649906,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-03 13:00:00,TSLA,118.32,121.90,116.61,118.93,2886437,118.9440,118.9440,118.676582,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1032,2023-12-08 13:00:00,TSLA,242.19,243.37,239.75,240.40,1406899,242.3970,241.1094,241.760952,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1033,2023-12-08 14:00:00,TSLA,240.46,243.65,239.27,242.40,15237079,242.3730,241.2916,241.821814,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1034,2023-12-08 15:00:00,TSLA,242.40,245.23,242.40,245.19,22432349,242.6320,241.5218,242.142593,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1035,2023-12-08 16:00:00,TSLA,245.19,245.27,241.67,241.97,15853691,242.5470,241.5982,242.126156,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Trying aggregate scores combinations

In [8]:
for col in score_cols:
    wrong_len = df[col].apply(lambda x: isinstance(x, list) and len(x) != 10)
    if wrong_len.any():
        print(f"{col} has lists with length != 10")
        print (len(wrong_len))

In [9]:
wrong_len

0       False
1       False
2       False
3       False
4       False
        ...  
1032    False
1033    False
1034    False
1035    False
1036    False
Name: scores_22, Length: 1037, dtype: bool

In [24]:
score_cols = [f"scores_{i}" for i in range(23)]

import ast

for col in score_cols:
    df[col] = df[col].apply(
        lambda x: [int(v) for v in ast.literal_eval(x)] if isinstance(x, str) else x
    )


## Aggregating Scores: Attempt 1
Sum element-wise across lists.

Normalize each summed value separately:

If positive → scale 0 → 1 based on max positive in the summed list

If negative → scale 0 → -1 based on min negative in the summed list

Zero stays zero.

Then compute average, max, min of the normalized list.

The key fix is that the divisor for normalization must be:

max_pos = max([x for x in summed if x>0], default=0)

min_neg = min([x for x in summed if x<0], default=0)

Each element is normalized based on its sign:

x > 0 → x / max_pos

x < 0 → x / abs(min_neg)

x == 0 → 0

This ensures:

Positive numbers map 0→1

Negative numbers map 0→-1

No value exceeds ±1

In [38]:
score_cols = [f"scores_{i}" for i in range(23)]
TARGET_LEN = 10

def sum_scores(row):
    lists = []
    for col in score_cols:
        val = row[col]
        if isinstance(val, list) and len(val) == TARGET_LEN:
            # convert string numbers to int
            lists.append([int(x) for x in val])
    if not lists:
        return [0]*TARGET_LEN
    return np.sum(np.array(lists, dtype=int), axis=0).tolist()

df["sum_of_scores"] = df.apply(sum_scores, axis=1)


In [ ]:
def sum_scores(row):
    lists = []
    for col in score_cols:
        val = row[col]
        if isinstance(val, list) and len(val) == TARGET_LEN:
            # convert string numbers to int
            lists.append([int(x) for x in val])
    if not lists:
        return [0]*TARGET_LEN
    return np.sum(np.array(lists, dtype=int), axis=0).tolist()

df["sum_of_scores"] = df.apply(sum_scores, axis=1)


In [40]:
def avg_scores(row):
    lists = []
    for col in score_cols:
        val = row[col]
        if isinstance(val, list) and len(val) == TARGET_LEN:
            lists.append([int(x) for x in val])
    if not lists:
        return [0]*TARGET_LEN
    arr = np.array(lists, dtype=int)
    return (np.sum(arr, axis=0) / len(lists)).tolist()

df["avg_of_scores"] = df.apply(avg_scores, axis=1)


In [43]:
df["normalized_avg"] = df["avg_of_scores"].apply(lambda lst: [x / 5 for x in lst])


In [46]:
#df = df.drop(columns=["normalized_list", "avg_normalized", "max_normalized", "min_normalized"])

In [48]:
# element-wise max
def max_scores(row):
    lists = []
    for col in score_cols:
        val = row[col]
        if isinstance(val, list) and len(val) == TARGET_LEN:
            lists.append([int(x) for x in val])
    if not lists:
        return [0]*TARGET_LEN
    arr = np.array(lists, dtype=int)
    return np.max(arr, axis=0).tolist()

df["max_of_scores"] = df.apply(max_scores, axis=1)

# element-wise min
def min_scores(row):
    lists = []
    for col in score_cols:
        val = row[col]
        if isinstance(val, list) and len(val) == TARGET_LEN:
            lists.append([int(x) for x in val])
    if not lists:
        return [0]*TARGET_LEN
    arr = np.array(lists, dtype=int)
    return np.min(arr, axis=0).tolist()

df["min_of_scores"] = df.apply(min_scores, axis=1)


In [50]:
df["normalized_min"] = df["min_of_scores"].apply(lambda lst: [x / 5 for x in lst])
df["normalized_max"] = df["max_of_scores"].apply(lambda lst: [x / 5 for x in lst])


In [53]:
df["avg_score_of_normalized_min"] = df["normalized_min"].apply(lambda lst: sum(lst)/len(lst) if lst else 0)
df["avg_score_of_normalized_max"] = df["normalized_max"].apply(lambda lst: sum(lst)/len(lst) if lst else 0)
df["avg_score_of_normalized_avg"] = df["normalized_avg"].apply(lambda lst: sum(lst)/len(lst) if lst else 0)


In [54]:
df.iloc[0]

timestamp                                                    2023-01-03 09:00:00
symbol                                                                      TSLA
open                                                                      120.02
high                                                                      121.23
low                                                                       115.94
close                                                                     118.52
volume                                                                   1306706
sma_10                                                                    118.52
sma_50                                                                    118.52
ema_20                                                                    118.52
rsi_14                                                                       0.0
macd                                                                         0.0
macd_signal                 

In [52]:
4/5

0.8

In [56]:
df.iloc[0]

timestamp                                                    2023-01-03 09:00:00
symbol                                                                      TSLA
open                                                                      120.02
high                                                                      121.23
low                                                                       115.94
close                                                                     118.52
volume                                                                   1306706
sma_10                                                                    118.52
sma_50                                                                    118.52
ema_20                                                                    118.52
rsi_14                                                                       0.0
macd                                                                         0.0
macd_signal                 

In [55]:
df.to_csv("brian_news_stocks_tesla_2023_processses_with_agg.csv", index=False)

# LSTM

## close price only

## Other financial indicators

## Close price with avg min, and max score of all. 